# Snowflake x GenAI: LLM Functions: Lesson 2: The TRANSLATE() Function

AI_TRANSLATE, reverse translation, and the legacy TRANSLATE functions.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `SXGA2_LLM_ADMIN` (ACCOUNTADMIN), `SXGA2_LLM_ROLE` (SYSADMIN) and `SXGA2_LLM_WH` (COMPUTE_WH).
- Course code that no longer works runs **as written** and reports its error; a **Substitute** cell with current functions follows it.
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- 🛠️ **Course update needed** marks what has to change in the course itself.
- Files the course has learners download are on `@UTIL_DB.PUBLIC.SXGA2_LLM_DOWNLOADS`, standing in for the learner's Downloads folder.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "SXGA2_LLM_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "SXGA2_LLM_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "SXGA2_LLM_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: Calling the Translate Function

> **Course:** 🧪 LAB: Call the Translate Function. Paste the code snippet into a worksheet and run it.

In [ ]:
%%sql -r translate_hello
select ai_translate('Hello', 'en', 'ja');

> **Course:** 🎯 CHALLENGE LAB: Reverse Translation. Take the value we just received and translate it back to English (LLMM_080).

In [ ]:
%%sql -r reverse_translate
-- Derived solution
select ai_translate('こんにちは', 'ja', 'en');

> ⚠️ **Audit note:** LLMM_080 shows `snowflake.cortex.translate(...)` (the legacy function). The course's own code on this page already uses `ai_translate`; the page is half-migrated.

## Page: Special TRANSLATE() Considerations

> **Course:** 🧪 LAB: Set the Worksheet Context (LLMM_085, 086): choose database SNOWFLAKE and schema CORTEX. Then run the three forms (fully qualified, partially qualified, not qualified). Only the first two call the Cortex function; the bare `translate(...)` is the plain string function that replaces characters.

In [ ]:
%%sql -r set_cortex_context
-- SQL equivalent of the Choose database drop-down
use schema snowflake.cortex;

> ⚠️ **Audit note:** Substitute cells use the current `AI_*` functions. The legacy `SNOWFLAKE.CORTEX.TRANSLATE`, `SENTIMENT` and `SUMMARIZE` are blocked on new accounts (verified 2026-11-14: "New accounts are blocked from using legacy SNOWFLAKE.CORTEX functions").

In [ ]:
%%sql -r translate_qualified_forms
execute immediate 'begin
  execute immediate ''select
    snowflake.cortex.translate(''''Goodbye!'''', ''''en'''', ''''fr'''')
    , cortex.translate(''''Goodbye!'''', ''''en'''', ''''fr'''')
    , translate(''''Goodbye!'''', ''''en'''', ''''fr'''')'';
  return ''ran without error'';
exception when other then
  return ''Error (as run today): '' || sqlerrm;
end;';

In [ ]:
%%sql -r translate_bare_string_function
-- LLMM_086 third column: the bare name resolves to the global string function TRANSLATE (character replacement), not an AI function
select translate('Goodbye!', 'en', 'fr') as not_qualified;

In [ ]:
%%sql -r translate_substitute_forms
-- Substitute: the current function. `ai_translate` is the same from any schema, so the "will not return" trap disappears
select ai_translate('Goodbye!', 'en', 'fr') as fully_qualified_equivalent,
       snowflake.cortex.ai_translate('Goodbye!', 'en', 'fr') as qualified;

> 🛠️ **Course update needed:** The whole 'bare TRANSLATE() is a different function' lesson only applies to the legacy name. With `AI_TRANSLATE` there is nothing to confuse; reword or drop the page and its quiz.

## Page: Reverse Translation to Check Quality

> **Course:** 🧪 LAB: Create a Variable and Test It (LLMM_086): `set my_text = 'My favorite color is blue.';` then `select $my_text;`.

In [ ]:
%%sql -r set_my_text
-- From screenshot LLMM_086
set my_text = 'My favorite color is blue.';

select $my_text;

> **Course:** 🧪 LAB: Call the Translate Function Using the Variable (the course's code, which uses `cortex.translate`).

In [ ]:
%%sql -r reverse_translation_as_given
execute immediate 'begin
  execute immediate ''select $my_text as OG_ENGLISH
    , cortex.translate($my_text, ''''en'''', ''''fr'''') as EN_2_FR
    , cortex.translate(EN_2_FR, ''''fr'''', ''''en'''') as EN_2_FR_2_EN
    , OG_ENGLISH = EN_2_FR_2_EN as Reverse_Translation_Success'';
  return ''ran without error'';
exception when other then
  return ''Error (as run today): '' || sqlerrm;
end;';

In [ ]:
%%sql -r reverse_translation_substitute
-- Substitute: the same query with ai_translate
select $my_text as OG_ENGLISH
    , ai_translate($my_text, 'en', 'fr') as EN_2_FR
    , ai_translate(EN_2_FR, 'fr', 'en') as EN_2_FR_2_EN
    , OG_ENGLISH = EN_2_FR_2_EN as Reverse_Translation_Success;

In [ ]:
%%sql -r reverse_translation_quote
-- Derived: the course suggests trying quotes; most fail the exact-match column but keep their meaning
set my_text = 'The best and most beautiful things in the world cannot be seen or even touched, they must be felt with the heart.';
select $my_text as OG_ENGLISH
    , ai_translate($my_text, 'en', 'fr') as EN_2_FR
    , ai_translate(EN_2_FR, 'fr', 'en') as EN_2_FR_2_EN
    , OG_ENGLISH = EN_2_FR_2_EN as Reverse_Translation_Success;

## Page: Checking the Quality of GenAI

Reading only (GenAI animation commentary).

> 🛠️ **Course update needed:** The animation commentary (Bing Image Creator, aitubo.com models) is dated and refers to a 'last month' state of the art.